In [ ]:
# ============================================================
# Cell 1: Install
# ============================================================
!pip -q install -U transformers accelerate langchain langchain-core langchain-community pandas numpy


In [ ]:
# ============================================================
# Cell 2: Imports + config
# ============================================================
import os, re, json, time, math
import numpy as np
import pandas as pd
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

from langchain_community.llms import HuggingFacePipeline
from langchain_core.prompts import (
    PromptTemplate,
    ChatPromptTemplate,
    FewShotPromptTemplate,
    FewShotChatMessagePromptTemplate,
)
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

str_parser = StrOutputParser()

# Evaluate on a small test-set for fast runs (increase later)
MAX_EXAMPLES_PER_TASK = 25

# Greedy decoding for deterministic evaluation
GEN_DEFAULTS = dict(
    max_new_tokens=128,
    do_sample=False,           # greedy
    return_full_text=False
)

# SmolLM2 instruct variants (IDs verified on Hugging Face)
SMOLLM_MODELS = [
    "HuggingFaceTB/SmolLM2-135M-Instruct",
    "HuggingFaceTB/SmolLM2-360M-Instruct",
    "HuggingFaceTB/SmolLM2-1.7B-Instruct",
]


In [ ]:
# ===============================
# Cell 3 (REPLACE): Model loader
# ===============================
def load_hf_llm(model_id: str, gen_kwargs=None):
    """
    Loads a causal LM + tokenizer and returns:
      - llm: LangChain HuggingFacePipeline (string in -> string out)
      - tokenizer: HF tokenizer
      - model: HF model
    Notes:
      - We pass generation kwargs via HuggingFacePipeline(pipeline_kwargs=...)
        to avoid LangChain default max_length=20.
    """
    gen_kwargs = gen_kwargs or {
        "max_new_tokens": 128,
        "do_sample": False,      # greedy for evaluation
        # temperature/top_p ignored when do_sample=False, but harmless
        "temperature": 0.0,
        "top_p": 1.0,
    }

    use_cuda = torch.cuda.is_available()
    dtype = torch.float16 if use_cuda else torch.float32

    tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=dtype,
        device_map="auto",
    )

    # Create pipeline WITHOUT generation params here
    gen_pipe = pipeline(
        task="text-generation",
        model=model,
        tokenizer=tokenizer,
        return_full_text=False,
    )

    # IMPORTANT: override LC defaults (no max_length)
    llm = HuggingFacePipeline(
        pipeline=gen_pipe,
        pipeline_kwargs=gen_kwargs
    )

    return llm, tokenizer, model


In [ ]:
# ============================================================
# Cell 4: Robust adapter: LangChain chat prompt -> HF prompt string
# ============================================================
def make_to_hf_prompt(tokenizer):
    """
    Returns a RunnableLambda that converts LangChain prompt output into
    a string formatted for the model. Works even if messages come in as:
      - ChatPromptValue (has .to_messages())
      - list of BaseMessage
      - list of (role, content) tuples
      - list of {role, content} dicts
    """
    def lc_messages_to_hf_chat(x):
        # 1) Normalize to list of messages
        if hasattr(x, "to_messages"):
            messages = x.to_messages()
        elif isinstance(x, dict) and "messages" in x:
            messages = x["messages"]
        else:
            messages = x

        # 2) Map roles to HF chat roles
        role_map = {
            "human": "user",
            "user": "user",
            "ai": "assistant",
            "assistant": "assistant",
            "system": "system",
        }

        chat = []
        for m in messages:
            if isinstance(m, tuple) and len(m) == 2:
                role, content = m
            elif isinstance(m, dict) and "role" in m and "content" in m:
                role, content = m["role"], m["content"]
            else:
                role = getattr(m, "type", None)
                content = getattr(m, "content", None)

            role = role_map.get(role, role)
            chat.append({"role": role, "content": content})

        # 3) Use tokenizer chat template if available
        if getattr(tokenizer, "chat_template", None):
            return tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)

        # Fallback for models without chat_template
        lines = [f"{t['role'].upper()}: {t['content']}" for t in chat]
        lines.append("ASSISTANT:")
        return "\n".join(lines)

    return RunnableLambda(lc_messages_to_hf_chat)


In [ ]:
# ============================================================
# Cell 5: Task datasets (small but clean for evaluation)
# ============================================================
TASKS = {}

# --- Task 1: Sentiment (3 classes)
TASKS["sentiment"] = [
    ("Loved it. Smooth experience and fast support.", "positive"),
    ("This is the worst update. Everything is broken.", "negative"),
    ("It works as expected. Nothing special.", "neutral"),
    ("Customer service was helpful and quick.", "positive"),
    ("Terrible performance; crashes every time.", "negative"),
    ("Not bad, but not amazing either.", "neutral"),
]

# --- Task 2: Topic (5 classes)
TOPICS = ["sports", "politics", "technology", "health", "finance"]
TASKS["topic"] = [
    ("The team won 3-1 and advanced to the finals.", "sports"),
    ("Parliament debated the new education bill today.", "politics"),
    ("The new GPU architecture improves training speed.", "technology"),
    ("Doctors recommend regular exercise to reduce risk.", "health"),
    ("Stocks fell after the earnings report missed estimates.", "finance"),
    ("The striker scored a hat-trick in the derby.", "sports"),
]

# --- Task 3: NLI (3 classes)
NLI_LABELS = ["entailment", "contradiction", "neutral"]
TASKS["nli"] = [
    (("A man is riding a bicycle.", "A person is on a bike."), "entailment"),
    (("The cat is sleeping on the couch.", "The cat is running outside."), "contradiction"),
    (("A woman is reading a book.", "A woman is in a library."), "neutral"),
    (("The meeting starts at 3 PM.", "The meeting is scheduled for 3 PM."), "entailment"),
]

# --- Task 4: Summarization (reference summaries)
TASKS["summarize"] = [
    (
        "We trained a small model on a filtered dataset. Validation loss decreased for 4 epochs, then plateaued. "
        "We suspect data leakage in one split. Next step: rebuild splits and re-evaluate.",
        "Validation improved then plateaued; suspected split leakage; rebuild splits and re-evaluate."
    ),
    (
        "The project is delayed due to missing dependencies and unstable builds. The team will pin versions, "
        "add CI checks, and run integration tests nightly.",
        "Project delay from dependencies/build instability; plan: pin versions, add CI, nightly integration tests."
    ),
]

# --- Task 5: Entity extraction -> JSON (people/orgs/dates)
TASKS["extract_json"] = [
    (
        "On 12 Jan 2026, Kajal Sharma met researchers from IIT Delhi to discuss a demo. A follow-up with Hugging Face was mentioned.",
        {"people": ["Kajal Sharma"], "orgs": ["IIT Delhi", "Hugging Face"], "dates": ["12 Jan 2026"]}
    ),
    (
        "Ravi Kumar emailed OpenAI on 3 March 2025 regarding access. The reply referenced Microsoft as a partner.",
        {"people": ["Ravi Kumar"], "orgs": ["OpenAI", "Microsoft"], "dates": ["3 March 2025"]}
    ),
]

# Cap size for speed
for k in TASKS:
    TASKS[k] = TASKS[k][:MAX_EXAMPLES_PER_TASK]


In [ ]:
# ============================================================
# Cell 6: Metrics
# ============================================================
def normalize_label(x: str) -> str:
    if x is None:
        return ""
    x = x.strip().lower()
    # keep only first token if model babbles
    x = re.split(r"[\s\.,;:\n]+", x)[0]
    return x

def accuracy(preds, golds):
    return float(np.mean([p == g for p, g in zip(preds, golds)])) if golds else 0.0

def macro_f1(preds, golds, labels):
    # simple macro-F1
    f1s = []
    for lab in labels:
        tp = sum((p == lab and g == lab) for p, g in zip(preds, golds))
        fp = sum((p == lab and g != lab) for p, g in zip(preds, golds))
        fn = sum((p != lab and g == lab) for p, g in zip(preds, golds))
        prec = tp / (tp + fp) if (tp + fp) else 0.0
        rec  = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = (2 * prec * rec / (prec + rec)) if (prec + rec) else 0.0
        f1s.append(f1)
    return float(np.mean(f1s)) if f1s else 0.0

def rouge_l(pred: str, ref: str) -> float:
    # ROUGE-L F1 via LCS
    pred_tokens = pred.strip().split()
    ref_tokens  = ref.strip().split()
    n, m = len(pred_tokens), len(ref_tokens)
    if n == 0 or m == 0:
        return 0.0

    # LCS DP (O(nm) - OK for short summaries)
    dp = [[0]*(m+1) for _ in range(n+1)]
    for i in range(n):
        for j in range(m):
            if pred_tokens[i] == ref_tokens[j]:
                dp[i+1][j+1] = dp[i][j] + 1
            else:
                dp[i+1][j+1] = max(dp[i][j+1], dp[i+1][j])

    lcs = dp[n][m]
    prec = lcs / n
    rec = lcs / m
    return (2*prec*rec/(prec+rec)) if (prec+rec) else 0.0

def parse_json_best_effort(s: str):
    s = s.strip()
    try:
        return json.loads(s)
    except Exception:
        m = re.search(r"\{.*\}", s, flags=re.DOTALL)
        if not m:
            return None
        try:
            return json.loads(m.group(0))
        except Exception:
            return None

def set_f1(pred_list, gold_list):
    p = set([x.strip().lower() for x in pred_list if isinstance(x, str)])
    g = set([x.strip().lower() for x in gold_list if isinstance(x, str)])
    if not p and not g:
        return 1.0
    if not p or not g:
        return 0.0
    tp = len(p & g)
    prec = tp / len(p) if p else 0.0
    rec  = tp / len(g) if g else 0.0
    return (2*prec*rec/(prec+rec)) if (prec+rec) else 0.0


In [ ]:
# ============================================================
# Cell 7: Define 5 template patterns (SAFE: braces escaped)
# ============================================================
def esc_curly(s: str) -> str:
    # Escape { } so LangChain's SECOND formatting pass doesn't treat JSON keys as variables
    return s.replace("{", "{{").replace("}", "}}")



TEMPLATES = ["T1_prompt", "T2_chat", "T3_fewshot_prompt", "T4_fewshot_chat", "T5_json"]

def build_templates():
    """
    Returns dict: templates[template_id][task_id] = prompt object
    NOTE: Any literal { } must be escaped as {{ }} in LangChain templates.
    """
    templates = {t: {} for t in TEMPLATES}

    # ------------------------------------------------------------
    # Task: sentiment (label only)
    # ------------------------------------------------------------
    templates["T1_prompt"]["sentiment"] = PromptTemplate.from_template(
        "Classify sentiment as one of: positive, negative, neutral.\n"
        "Output ONLY the label.\n\nTEXT:\n{text}\nLABEL:"
    )

    templates["T2_chat"]["sentiment"] = ChatPromptTemplate.from_messages([
        ("system", "You are a strict sentiment classifier. Output only: positive, negative, or neutral."),
        ("human", "TEXT:\n{text}\nLABEL:")
    ])

    sent_examples = [
        {"text": "Loved it. Smooth experience.", "label": "positive"},
        {"text": "Worst update. Everything broke.", "label": "negative"},
        {"text": "It works. Nothing special.", "label": "neutral"},
    ]
    sent_example_prompt = PromptTemplate.from_template("Text: {text}\nSentiment: {label}")

    templates["T3_fewshot_prompt"]["sentiment"] = FewShotPromptTemplate(
        examples=sent_examples,
        example_prompt=sent_example_prompt,
        prefix="Classify sentiment. Labels: positive, negative, neutral.\nExamples:",
        suffix="Text: {text}\nSentiment:",
        input_variables=["text"],
    )

    sent_chat_examples = [
        {"input": "Loved it. Smooth experience.", "output": "positive"},
        {"input": "Worst update. Everything broke.", "output": "negative"},
        {"input": "It works. Nothing special.", "output": "neutral"},
    ]
    sent_chat_ex_prompt = ChatPromptTemplate.from_messages([("human","{input}"),("ai","{output}")])
    templates["T4_fewshot_chat"]["sentiment"] = ChatPromptTemplate.from_messages([
        ("system", "Output only: positive, negative, neutral."),
        FewShotChatMessagePromptTemplate(example_prompt=sent_chat_ex_prompt, examples=sent_chat_examples),
        ("human", "{text}")
    ])

    # IMPORTANT: escape JSON braces
    templates["T5_json"]["sentiment"] = PromptTemplate.from_template(
        "Classify sentiment as positive, negative, or neutral.\n"
        "Return ONLY valid JSON like: {{\"label\": \"positive\"}}\n\n"
        "TEXT:\n{text}\nJSON:"
    )

    # ------------------------------------------------------------
    # Task: topic (5 labels)
    # ------------------------------------------------------------
    templates["T1_prompt"]["topic"] = PromptTemplate.from_template(
        f"Classify topic as one of: {', '.join(TOPICS)}.\n"
        "Output ONLY the label.\n\nTEXT:\n{text}\nLABEL:"
    )

    templates["T2_chat"]["topic"] = ChatPromptTemplate.from_messages([
        ("system", f"You are a strict topic classifier. Output only one label from: {', '.join(TOPICS)}."),
        ("human", "TEXT:\n{text}\nLABEL:")
    ])

    topic_examples = [
        {"text": "The striker scored twice and won the match.", "label":"sports"},
        {"text": "New legislation was introduced in parliament.", "label":"politics"},
        {"text": "A new transformer architecture was released.", "label":"technology"},
        {"text": "Doctors advise sleep and balanced nutrition.", "label":"health"},
        {"text": "Markets rallied after strong earnings.", "label":"finance"},
    ]
    topic_example_prompt = PromptTemplate.from_template("Text: {text}\nTopic: {label}")
    templates["T3_fewshot_prompt"]["topic"] = FewShotPromptTemplate(
        examples=topic_examples,
        example_prompt=topic_example_prompt,
        prefix=f"Classify topic. Labels: {', '.join(TOPICS)}.\nExamples:",
        suffix="Text: {text}\nTopic:",
        input_variables=["text"],
    )

    topic_chat_examples = [
        {"input": "Markets rallied after strong earnings.", "output":"finance"},
        {"input": "Doctors advise sleep and balanced nutrition.", "output":"health"},
        {"input": "Parliament debated the bill.", "output":"politics"},
    ]
    topic_chat_ex_prompt = ChatPromptTemplate.from_messages([("human","{input}"),("ai","{output}")])
    templates["T4_fewshot_chat"]["topic"] = ChatPromptTemplate.from_messages([
        ("system", f"Output only one label from: {', '.join(TOPICS)}."),
        FewShotChatMessagePromptTemplate(example_prompt=topic_chat_ex_prompt, examples=topic_chat_examples),
        ("human", "{text}")
    ])

    templates["T5_json"]["topic"] = PromptTemplate.from_template(
        f"Classify topic as one of: {', '.join(TOPICS)}.\n"
        "Return ONLY valid JSON like: {{\"label\": \"technology\"}}\n\n"
        "TEXT:\n{text}\nJSON:"
    )

    # ------------------------------------------------------------
    # Task: NLI (3 labels)
    # ------------------------------------------------------------
    templates["T1_prompt"]["nli"] = PromptTemplate.from_template(
        "Natural Language Inference.\n"
        "Given Premise and Hypothesis, output ONLY one label: entailment, contradiction, neutral.\n\n"
        "PREMISE:\n{premise}\n\nHYPOTHESIS:\n{hypothesis}\n\nLABEL:"
    )

    templates["T2_chat"]["nli"] = ChatPromptTemplate.from_messages([
        ("system", "You do NLI. Output only: entailment, contradiction, or neutral."),
        ("human", "PREMISE:\n{premise}\n\nHYPOTHESIS:\n{hypothesis}\n\nLABEL:")
    ])

    nli_examples = [
        {"premise":"A man is riding a bicycle.", "hypothesis":"A person is on a bike.", "label":"entailment"},
        {"premise":"The cat is sleeping.", "hypothesis":"The cat is running outside.", "label":"contradiction"},
        {"premise":"A woman is reading a book.", "hypothesis":"A woman is in a library.", "label":"neutral"},
    ]
    nli_example_prompt = PromptTemplate.from_template(
        "Premise: {premise}\nHypothesis: {hypothesis}\nLabel: {label}"
    )
    templates["T3_fewshot_prompt"]["nli"] = FewShotPromptTemplate(
        examples=nli_examples,
        example_prompt=nli_example_prompt,
        prefix="Do NLI. Labels: entailment, contradiction, neutral.\nExamples:",
        suffix="Premise: {premise}\nHypothesis: {hypothesis}\nLabel:",
        input_variables=["premise","hypothesis"],
    )

    nli_chat_examples = [
        {"input":"Premise: A man is riding a bicycle.\nHypothesis: A person is on a bike.", "output":"entailment"},
        {"input":"Premise: The cat is sleeping.\nHypothesis: The cat is running outside.", "output":"contradiction"},
    ]
    nli_chat_ex_prompt = ChatPromptTemplate.from_messages([("human","{input}"),("ai","{output}")])
    templates["T4_fewshot_chat"]["nli"] = ChatPromptTemplate.from_messages([
        ("system", "Output only: entailment, contradiction, neutral."),
        FewShotChatMessagePromptTemplate(example_prompt=nli_chat_ex_prompt, examples=nli_chat_examples),
        ("human", "Premise: {premise}\nHypothesis: {hypothesis}")
    ])

    templates["T5_json"]["nli"] = PromptTemplate.from_template(
        "Do NLI. Labels: entailment, contradiction, neutral.\n"
        "Return ONLY valid JSON like: {{\"label\": \"neutral\"}}\n\n"
        "PREMISE:\n{premise}\n\nHYPOTHESIS:\n{hypothesis}\n\nJSON:"
    )

    # ------------------------------------------------------------
    # Task: summarize
    # ------------------------------------------------------------
    templates["T1_prompt"]["summarize"] = PromptTemplate.from_template(
        "Summarize the text in ONE sentence.\n\nTEXT:\n{text}\n\nSUMMARY:"
    )

    templates["T2_chat"]["summarize"] = ChatPromptTemplate.from_messages([
        ("system", "Summarize in one sentence. Do not add facts."),
        ("human", "TEXT:\n{text}\nSUMMARY:")
    ])

    sum_examples = [
        {"text":"Training improved then plateaued; suspected leakage; rebuild splits.", "label":"Validation plateaued; suspected leakage; rebuild splits."},
        {"text":"Builds unstable due to dependencies; plan: pin versions and add CI.", "label":"Dependencies caused instability; plan to pin versions and add CI."},
    ]
    sum_example_prompt = PromptTemplate.from_template("Text: {text}\nSummary: {label}")
    templates["T3_fewshot_prompt"]["summarize"] = FewShotPromptTemplate(
        examples=sum_examples,
        example_prompt=sum_example_prompt,
        prefix="Summarize in one sentence. Examples:",
        suffix="Text: {text}\nSummary:",
        input_variables=["text"],
    )

    sum_chat_examples = [
        {"input":"TEXT: The project is delayed due to unstable builds.", "output":"The project is delayed because builds are unstable."},
        {"input":"TEXT: Validation improved then plateaued; suspected leakage.", "output":"Validation improved then plateaued, likely due to leakage."},
    ]
    sum_chat_ex_prompt = ChatPromptTemplate.from_messages([("human","{input}"),("ai","{output}")])
    templates["T4_fewshot_chat"]["summarize"] = ChatPromptTemplate.from_messages([
        ("system", "One-sentence summary. No extra facts."),
        FewShotChatMessagePromptTemplate(example_prompt=sum_chat_ex_prompt, examples=sum_chat_examples),
        ("human", "TEXT:\n{text}")
    ])

    templates["T5_json"]["summarize"] = PromptTemplate.from_template(
        "Summarize in one sentence.\n"
        "Return ONLY valid JSON like: {{\"summary\": \"...\"}}\n\n"
        "TEXT:\n{text}\nJSON:"
    )

    # ------------------------------------------------------------
    # Task: extract_json (IMPORTANT: no literal {people...} in chat)
    # ------------------------------------------------------------
    templates["T1_prompt"]["extract_json"] = PromptTemplate.from_template(
        "Extract entities from the text.\n"
        "Return ONLY valid JSON with keys: people (list), orgs (list), dates (list).\n\n"
        "TEXT:\n{text}\nJSON:"
    )

    # Avoid braces in system message to prevent accidental variables
    templates["T2_chat"]["extract_json"] = ChatPromptTemplate.from_messages([
        ("system", "Extract entities. Output ONLY valid JSON with keys: people, orgs, dates."),
        ("human", "TEXT:\n{text}\nJSON:")
    ])

    ex_examples = [
    {
        "text": "On 12 Jan 2026, Kajal Sharma met IIT Delhi.",
        "label": esc_curly('{"people":["Kajal Sharma"],"orgs":["IIT Delhi"],"dates":["12 Jan 2026"]}')
    },
    {
        "text": "Ravi Kumar emailed OpenAI on 3 March 2025.",
        "label": esc_curly('{"people":["Ravi Kumar"],"orgs":["OpenAI"],"dates":["3 March 2025"]}')
    },
    ]

    ex_example_prompt = PromptTemplate.from_template("Text: {text}\nJSON: {label}")
    templates["T3_fewshot_prompt"]["extract_json"] = FewShotPromptTemplate(
        examples=ex_examples,
        example_prompt=ex_example_prompt,
        prefix="Extract people/orgs/dates and return ONLY JSON. Examples:",
        suffix="Text: {text}\nJSON:",
        input_variables=["text"],
    )

    ex_chat_examples = [
        {"input":"TEXT: On 12 Jan 2026, Kajal Sharma met IIT Delhi.", "output":'{"people":["Kajal Sharma"],"orgs":["IIT Delhi"],"dates":["12 Jan 2026"]}'},
    ]
    ex_chat_ex_prompt = ChatPromptTemplate.from_messages([("human","{input}"),("ai","{output}")])
    templates["T4_fewshot_chat"]["extract_json"] = ChatPromptTemplate.from_messages([
        ("system", "Return ONLY valid JSON with keys: people, orgs, dates."),
        FewShotChatMessagePromptTemplate(example_prompt=ex_chat_ex_prompt, examples=ex_chat_examples),
        ("human", "TEXT:\n{text}\nJSON:")
    ])

    templates["T5_json"]["extract_json"] = PromptTemplate.from_template(
        "Extract entities.\n"
        "Return ONLY valid JSON with keys: people, orgs, dates.\n\n"
        "TEXT:\n{text}\nJSON:"
    )

    return templates

TEMPLATE_MAP = build_templates()
print("Built TEMPLATE_MAP safely.")


Built TEMPLATE_MAP safely.


In [ ]:
# ============================================================
# Sanity check: each prompt should require only the variables we pass
# ============================================================
TASK_IDS = ["sentiment", "topic", "nli", "summarize", "extract_json"]

for tid in TEMPLATES:
    for task in TASK_IDS:
        p = TEMPLATE_MAP[tid][task]
        vars_ = getattr(p, "input_variables", None)
        if vars_ is None:
            vars_ = []
        extra = set(vars_) - set(["text", "premise", "hypothesis"])
        if extra:
            print("BAD:", tid, task, "vars=", vars_)
print("Done.")


Done.


In [ ]:
# ============================================================
# Cell 8: Build chain for template + task
# ============================================================
def build_chain(llm, tokenizer, template_id: str, task_id: str):
    prompt_obj = TEMPLATE_MAP[template_id][task_id]

    # If chat prompt -> convert to HF chat string first
    if isinstance(prompt_obj, ChatPromptTemplate):
        to_hf = make_to_hf_prompt(tokenizer)
        chain = prompt_obj | to_hf | llm | str_parser
    else:
        chain = prompt_obj | llm | str_parser

    return chain


In [ ]:
# ============================================================
# Cell 9: Run one (template, task) and compute metrics
# ============================================================
def run_task_eval(chain, task_id: str, data):
    preds = []
    golds = []
    latencies = []

    for item in data:
        t0 = time.perf_counter()

        if task_id in ["sentiment", "topic"]:
            text, gold = item
            out = chain.invoke({"text": text})
            pred = normalize_label(out)

        elif task_id == "nli":
            (premise, hypothesis), gold = item
            out = chain.invoke({"premise": premise, "hypothesis": hypothesis})
            pred = normalize_label(out)

        elif task_id == "summarize":
            text, gold = item
            out = chain.invoke({"text": text}).strip()
            pred = out

        elif task_id == "extract_json":
            text, gold = item
            out = chain.invoke({"text": text}).strip()
            parsed = parse_json_best_effort(out)
            pred = parsed if parsed is not None else {}

        else:
            raise ValueError("Unknown task_id")

        t1 = time.perf_counter()
        latencies.append((t1 - t0) * 1000.0)  # ms

        preds.append(pred)
        golds.append(gold)

    # Compute metrics per task
    metrics = {}
    if task_id == "sentiment":
        metrics["acc"] = accuracy(preds, golds)
        metrics["macro_f1"] = macro_f1(preds, golds, ["positive","negative","neutral"])

    elif task_id == "topic":
        metrics["acc"] = accuracy(preds, golds)
        metrics["macro_f1"] = macro_f1(preds, golds, TOPICS)

    elif task_id == "nli":
        metrics["acc"] = accuracy(preds, golds)
        metrics["macro_f1"] = macro_f1(preds, golds, NLI_LABELS)

    elif task_id == "summarize":
        scores = [rouge_l(p, g) for p, g in zip(preds, golds)]
        metrics["rougeL_f1"] = float(np.mean(scores)) if scores else 0.0

    elif task_id == "extract_json":
        # set-F1 averaged over keys
        key_scores = []
        for p, g in zip(preds, golds):
            if not isinstance(p, dict):
                p = {}
            ks = []
            for key in ["people","orgs","dates"]:
                ks.append(set_f1(p.get(key, []), g.get(key, [])))
            key_scores.append(float(np.mean(ks)))
        metrics["entity_set_f1"] = float(np.mean(key_scores)) if key_scores else 0.0

    metrics["avg_latency_ms"] = float(np.mean(latencies)) if latencies else 0.0
    return metrics, preds


In [ ]:
# ============================================================
# Cell 10: 5x5 evaluation for one model
# ============================================================

from tqdm.auto import tqdm
import time, re, json
import numpy as np

def run_task_eval(chain, task_id: str, data, show_item_progress=False, bar_position=1):
    """
    Runs inference over 'data' and returns (metrics, preds).
    If show_item_progress=True, shows per-example tqdm.
    """
    preds, golds, latencies = [], [], []

    it = data
    if show_item_progress:
        it = tqdm(
            data,
            total=len(data),
            desc=f"  items | {task_id}",
            position=bar_position,
            leave=False,
            dynamic_ncols=True,
        )

    for item in it:
        t0 = time.perf_counter()

        if task_id in ["sentiment", "topic"]:
            text, gold = item
            out = chain.invoke({"text": text})
            pred = normalize_label(out)

        elif task_id == "nli":
            (premise, hypothesis), gold = item
            out = chain.invoke({"premise": premise, "hypothesis": hypothesis})
            pred = normalize_label(out)

        elif task_id == "summarize":
            text, gold = item
            pred = chain.invoke({"text": text}).strip()

        elif task_id == "extract_json":
            text, gold = item
            out = chain.invoke({"text": text}).strip()
            parsed = parse_json_best_effort(out)
            pred = parsed if parsed is not None else {}

        else:
            raise ValueError(f"Unknown task_id: {task_id}")

        t1 = time.perf_counter()
        latencies.append((t1 - t0) * 1000.0)

        preds.append(pred)
        golds.append(gold)

    # ---- metrics ----
    metrics = {}
    if task_id == "sentiment":
        metrics["acc"] = accuracy(preds, golds)
        metrics["macro_f1"] = macro_f1(preds, golds, ["positive","negative","neutral"])

    elif task_id == "topic":
        metrics["acc"] = accuracy(preds, golds)
        metrics["macro_f1"] = macro_f1(preds, golds, TOPICS)

    elif task_id == "nli":
        metrics["acc"] = accuracy(preds, golds)
        metrics["macro_f1"] = macro_f1(preds, golds, NLI_LABELS)

    elif task_id == "summarize":
        scores = [rouge_l(p, g) for p, g in zip(preds, golds)]
        metrics["rougeL_f1"] = float(np.mean(scores)) if scores else 0.0

    elif task_id == "extract_json":
        key_scores = []
        for p, g in zip(preds, golds):
            if not isinstance(p, dict):
                p = {}
            ks = []
            for key in ["people","orgs","dates"]:
                ks.append(set_f1(p.get(key, []), g.get(key, [])))
            key_scores.append(float(np.mean(ks)))
        metrics["entity_set_f1"] = float(np.mean(key_scores)) if key_scores else 0.0

    metrics["avg_latency_ms"] = float(np.mean(latencies)) if latencies else 0.0
    return metrics, preds

from tqdm.auto import tqdm
import pandas as pd

TASK_IDS = ["sentiment", "topic", "nli", "summarize", "extract_json"]

def evaluate_model_5x5(model_id: str, show_progress=True, show_item_progress=True):
    llm, tokenizer, _ = load_hf_llm(model_id)

    rows = []
    iterator = [(template_id, task_id) for template_id in TEMPLATES for task_id in TASK_IDS]

    if show_progress:
        pbar = tqdm(
            iterator,
            total=len(iterator),
            desc=f"combos | {model_id}",
            position=0,
            leave=True,
            dynamic_ncols=True,
        )
    else:
        pbar = iterator

    for template_id, task_id in pbar:
        if show_progress:
            pbar.set_postfix_str(f"{template_id} | {task_id}")

        chain = build_chain(llm, tokenizer, template_id, task_id)

        metrics, _ = run_task_eval(
            chain,
            task_id,
            TASKS[task_id],
            show_item_progress=show_item_progress,
            bar_position=1,   # inner bar below outer bar
        )

        rows.append({
            "model": model_id,
            "template": template_id,
            "task": task_id,
            **metrics
        })

    return pd.DataFrame(rows)


In [ ]:
# ============================================================
# Cell 11: Run all models (may take time for 1.7B)
# ============================================================
all_results = []

for mid in tqdm(SMOLLM_MODELS, desc="models", position=0, leave=True, dynamic_ncols=True):
    df = evaluate_model_5x5(mid, show_progress=True, show_item_progress=True)
    all_results.append(df)

results = pd.concat(all_results, ignore_index=True)
results


models:   0%|          | 0/3 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

/tmp/ipython-input-451396752.py:41: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(


combos | HuggingFaceTB/SmolLM2-135M-Instruct:   0%|          | 0/25 [00:00<?, ?it/s]

...

  items | summarize:   0%|          | 0/2 [00:00<?, ?it/s]

Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  items | extract_json:   0%|          | 0/2 [00:00<?, ?it/s]

Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


,model,template,task,acc,macro_f1,avg_latency_ms,rougeL_f1,entity_set_f1
0,HuggingFaceTB/SmolLM2-135M-Instruct,T1_prompt,sentiment,0.666667,0.555556,7993.848119,NaN,NaN
1,HuggingFaceTB/SmolLM2-135M-Instruct,T1_prompt,topic,0.166667,0.133333,3384.541374,NaN,NaN
2,HuggingFaceTB/SmolLM2-135M-Instruct,T1_prompt,nli,0.250000,0.133333,357.079732,NaN,NaN
3,HuggingFaceTB/SmolLM2-135M-Instruct,T1_prompt,summarize,NaN,NaN,1354.232717,0.130330,NaN
4,HuggingFaceTB/SmolLM2-135M-Instruct,T1_prompt,extract_json,NaN,NaN,4964.301898,NaN,0.0
...,...,...,...,...,...,...,...,...
70,HuggingFaceTB/SmolLM2-1.7B-Instruct,T5_json,sentiment,0.000000,0.000000,3336.480072,NaN,NaN
71,HuggingFaceTB/SmolLM2-1.7B-Instruct,T5_json,topic,0.000000,0.000000,3358.272614,NaN,NaN
72,HuggingFaceTB/SmolLM2-1.7B-Instruct,T5_json,nli,0.000000,0.000000,3294.364298,NaN,NaN
73,HuggingFaceTB/SmolLM2-1.7B-Instruct,T5_json,summarize,NaN,NaN,3372.482431,0.098701,NaN


In [ ]:
# ============================================================
# Cell 12: Summaries / pivots
# ============================================================
def pick_primary_metric(task):
    if task in ["sentiment","topic","nli"]:
        return "acc"
    if task == "summarize":
        return "rougeL_f1"
    if task == "extract_json":
        return "entity_set_f1"
    return None

results["primary_metric"] = results["task"].apply(pick_primary_metric)
results["score"] = results.apply(lambda r: r.get(r["primary_metric"], np.nan), axis=1)

# Best template per (model, task)
best = (results.sort_values("score", ascending=False)
              .groupby(["model","task"], as_index=False)
              .first()[["model","task","template","primary_metric","score","avg_latency_ms"]])

print("\nBest template per task (per model):")
display(best)

# Pivot: score by task vs model (best template only)
best_pivot = best.pivot(index="task", columns="model", values="score")
print("\nScore comparison (higher is better):")
display(best_pivot)

# Pivot: latency by task vs model (best template only)
lat_pivot = best.pivot(index="task", columns="model", values="avg_latency_ms")
print("\nLatency comparison (ms, lower is better):")
display(lat_pivot)



Best template per task (per model):


,model,task,template,primary_metric,score,avg_latency_ms
0,HuggingFaceTB/SmolLM2-1.7B-Instruct,extract_json,T4_fewshot_chat,entity_set_f1,0.888889,862.418523
1,HuggingFaceTB/SmolLM2-1.7B-Instruct,nli,T3_fewshot_prompt,acc,0.750000,2660.043245
2,HuggingFaceTB/SmolLM2-1.7B-Instruct,sentiment,T4_fewshot_chat,acc,1.000000,72.799982
3,HuggingFaceTB/SmolLM2-1.7B-Instruct,summarize,T3_fewshot_prompt,rougeL_f1,0.375000,786.713500
4,HuggingFaceTB/SmolLM2-1.7B-Instruct,topic,T4_fewshot_chat,acc,1.000000,86.497621
5,HuggingFaceTB/SmolLM2-135M-Instruct,extract_json,T4_fewshot_chat,entity_set_f1,0.444444,1509.977016
6,HuggingFaceTB/SmolLM2-135M-Instruct,nli,T3_fewshot_prompt,acc,0.750000,4842.244927
7,HuggingFaceTB/SmolLM2-135M-Instruct,sentiment,T3_fewshot_prompt,acc,0.833333,4949.408167
8,HuggingFaceTB/SmolLM2-135M-Instruct,summarize,T3_fewshot_prompt,rougeL_f1,0.330330,1272.415772
9,HuggingFaceTB/SmolLM2-135M-Instruct,topic,T3_fewshot_prompt,acc,1.000000,6033.006850



Score comparison (higher is better):


model,HuggingFaceTB/SmolLM2-1.7B-Instruct,HuggingFaceTB/SmolLM2-135M-Instruct,HuggingFaceTB/SmolLM2-360M-Instruct
task,,,
extract_json,0.888889,0.444444,0.444444
nli,0.750000,0.750000,0.500000
sentiment,1.000000,0.833333,1.000000
summarize,0.375000,0.330330,0.287879
topic,1.000000,1.000000,1.000000



Latency comparison (ms, lower is better):


model,HuggingFaceTB/SmolLM2-1.7B-Instruct,HuggingFaceTB/SmolLM2-135M-Instruct,HuggingFaceTB/SmolLM2-360M-Instruct
task,,,
extract_json,862.418523,1509.977016,1267.274481
nli,2660.043245,4842.244927,5226.854298
sentiment,72.799982,4949.408167,137.753757
summarize,786.713500,1272.415772,1004.392770
topic,86.497621,6033.006850,5383.366547


In [ ]:
# ============================================================
# Cell 13: 5x5 grid (template x task) for one selected model
# ============================================================
SELECT_MODEL = SMOLLM_MODELS[1]  # default: 360M

sub = results[results["model"] == SELECT_MODEL].copy()

grid = sub.pivot(index="template", columns="task", values="score")
print(f"\n5x5 grid scores for {SELECT_MODEL} (score metric depends on task):")
display(grid)



5x5 grid scores for HuggingFaceTB/SmolLM2-360M-Instruct (score metric depends on task):


task,extract_json,nli,sentiment,summarize,topic
template,,,,,
T1_prompt,0.000000,0.00,0.000000,0.108108,0.500000
T2_chat,0.000000,0.00,0.500000,0.227156,0.833333
T3_fewshot_prompt,0.000000,0.50,0.833333,0.287879,1.000000
T4_fewshot_chat,0.444444,0.25,1.000000,0.230059,0.666667
T5_json,0.000000,0.00,0.000000,0.038835,0.000000


In [ ]:
# ============================================================
# Cell 14: Inspect predictions for a particular (task, template, model)
# ============================================================
def inspect_one(model_id, template_id, task_id, n=5):
    llm, tokenizer, _ = load_hf_llm(model_id)
    chain = build_chain(llm, tokenizer, template_id, task_id)

    data = TASKS[task_id]
    metrics, preds = run_task_eval(chain, task_id, data)

    print("Metrics:", metrics)
    for i in range(min(n, len(data))):
        print("\n--- Example", i, "---")
        print("GOLD:", data[i][1])
        print("PRED:", preds[i])
        print("TEXT:", data[i][0] if task_id != "nli" else data[i][0])

# Example:
inspect_one(SMOLLM_MODELS[2], "T4_fewshot_chat", "sentiment", n=3)


Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

Metrics: {'acc': 1.0, 'macro_f1': 1.0, 'avg_latency_ms': 90.35943900031877}

--- Example 0 ---
GOLD: positive
PRED: positive
TEXT: Loved it. Smooth experience and fast support.

--- Example 1 ---
GOLD: negative
PRED: negative
TEXT: This is the worst update. Everything is broken.

--- Example 2 ---
GOLD: neutral
PRED: neutral
TEXT: It works as expected. Nothing special.
